
# Data Integration and Database Preparation

**Course:** BSMM 8730 - Data Acquisition and Management

**Project:** SmartCentres REIT Data Pipeline and Investment Analysis

**Group:** 2

## Objective

This notebook integrates cleaned financial, market,
and macroeconomic datasets to support relational
database design and SQL-based investment analysis.

## Scope

- Financial data integration
- Financial metric harmonization
- Temporal data alignment
- Database table preparation
- Data quality validation
- Export of database-ready datasets


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("default")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# Define project paths

BASE_DIR = Path.home() / "Desktop" / "8730 _ Group Assignment"

PROCESSED_DIR = BASE_DIR / "processed"

INTEGRATED_DIR = BASE_DIR / "integrated"

assert PROCESSED_DIR.is_dir(), "Processed folder not found."

INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Integrated directory:", INTEGRATED_DIR)
print("Processed directory exists:", PROCESSED_DIR.exists())
print("Integrated directory exists:", INTEGRATED_DIR.exists())

Project directory: /Users/yuanquan0901/Desktop/8730 _ Group Assignment
Processed directory: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/processed
Integrated directory: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/integrated
Processed directory exists: True
Integrated directory exists: True


In [6]:
# List all processed CSV files

csv_files = sorted(PROCESSED_DIR.glob("*.csv"))

file_summary = []

for file in csv_files:
    df = pd.read_csv(file)

    file_summary.append({
        "File": file.name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

file_summary_df = pd.DataFrame(file_summary)

display(file_summary_df)

print("Total CSV files:", len(csv_files))

,File,Rows,Columns
0,cpi_cleaned.csv,45,7
1,interest_rates_cleaned.csv,973,7
2,interest_rates_observed.csv,973,4
3,market_data_cleaned.csv,3764,13
4,peer_annual_financials_cleaned.csv,177,14
5,peer_quarterly_financials_cleaned.csv,105,14
6,retail_sales_cleaned.csv,45,11
7,smartcentres_annual_financials_cleaned.csv,134,14
8,smartcentres_quarterly_financials_cleaned.csv,36,14


Total CSV files: 9


## 2. Financial Data Integration

Combine the annual and quarterly financial datasets of
SmartCentres, RioCan, and Choice Properties into a unified
financial dataset while preserving reporting periods,
measurement units, and source references.

In [9]:
# Load cleaned financial datasets

sc_annual = pd.read_csv(
    PROCESSED_DIR / "smartcentres_annual_financials_cleaned.csv"
)

sc_quarterly = pd.read_csv(
    PROCESSED_DIR / "smartcentres_quarterly_financials_cleaned.csv"
)

peer_annual = pd.read_csv(
    PROCESSED_DIR / "peer_annual_financials_cleaned.csv"
)

peer_quarterly = pd.read_csv(
    PROCESSED_DIR / "peer_quarterly_financials_cleaned.csv"
)

financial_datasets = {
    "SmartCentres Annual": sc_annual,
    "SmartCentres Quarterly": sc_quarterly,
    "Peer Annual": peer_annual,
    "Peer Quarterly": peer_quarterly
}

for name, df in financial_datasets.items():
    print(name, df.shape)

SmartCentres Annual (134, 14)
SmartCentres Quarterly (36, 14)
Peer Annual (177, 14)
Peer Quarterly (105, 14)


In [11]:
# Validate column consistency

reference_columns = set(sc_annual.columns)

for name, df in financial_datasets.items():

    missing = reference_columns - set(df.columns)
    extra = set(df.columns) - reference_columns

    print(name)
    print("Missing columns:", missing)
    print("Extra columns:", extra)
    print("Schema consistent:", not missing and not extra)
    print("-" * 50)

display(pd.DataFrame({
    "Column": sc_annual.columns
}))

SmartCentres Annual
Missing columns: set()
Extra columns: set()
Schema consistent: True
--------------------------------------------------
SmartCentres Quarterly
Missing columns: set()
Extra columns: set()
Schema consistent: True
--------------------------------------------------
Peer Annual
Missing columns: set()
Extra columns: set()
Schema consistent: True
--------------------------------------------------
Peer Quarterly
Missing columns: set()
Extra columns: set()
Schema consistent: True
--------------------------------------------------


,Column
0,company
1,ticker
2,period_end
3,period_type
4,metric
5,value
6,unit
7,source_file
8,source_page
9,source_label


In [13]:
# Combine all financial datasets

assert all(
    set(df.columns) == reference_columns
    for df in financial_datasets.values()
), "Financial dataset schemas do not match."

financial_integrated = pd.concat(
    [
        sc_annual,
        sc_quarterly,
        peer_annual,
        peer_quarterly
    ],
    ignore_index=True
)

print("Integrated financial dataset shape:")
print(financial_integrated.shape)

display(financial_integrated.head(10))

Integrated financial dataset shape:
(452, 14)


,company,ticker,period_end,period_type,metric,value,unit,source_file,source_page,source_label,original_value,original_unit,extraction_method,original_line
0,SmartCentres,SRU-UN.TO,2023-12-31,FY,adjusted_debt_to_adjusted_ebitda,9.600000,Times,2024-SmartCentres-REIT-Annual-Report-.pdf,69,Adjusted Debt to Adjusted EBITDA,9.60,times,Manual transcription from extracted text,NaN
1,SmartCentres,SRU-UN.TO,2023-12-31,FY,affo,354.424000,CAD million,2024-SmartCentres-REIT-Annual-Report-.pdf,39,AFFO,354424.00,CAD thousand,Manual transcription from extracted text,NaN
2,SmartCentres,SRU-UN.TO,2023-12-31,FY,affo_payout_ratio,93.000000,Percent,2024-SmartCentres-REIT-Annual-Report-.pdf,39,Payout Ratio to AFFO,93.00,percent,Manual transcription from extracted text,NaN
3,SmartCentres,SRU-UN.TO,2023-12-31,FY,affo_per_unit_diluted,1.970000,CAD/unit,2024-SmartCentres-REIT-Annual-Report-.pdf,39,AFFO - diluted,1.97,CAD per unit,Manual transcription from extracted text,NaN
4,SmartCentres,SRU-UN.TO,2023-12-31,FY,affo_with_adjustments,330.051000,CAD million,2024-SmartCentres-REIT-Annual-Report-.pdf,39,AFFO with adjustments,330051.00,CAD thousand,Manual transcription from extracted text,NaN
5,SmartCentres,SRU-UN.TO,2023-12-31,FY,book_debt_to_assets_ratio,41.993656,Percent,2024-SmartCentres-REIT-Annual-Report-.pdf,92,total_debt / total_assets * 100,NaN,NaN,Calculated,NaN
6,SmartCentres,SRU-UN.TO,2023-12-31,FY,cash_and_cash_equivalents,34.743000,CAD million,2024-SmartCentres-REIT-Annual-Report-.pdf,92,Cash and cash equivalents,34743.00,CAD thousand,Text-based extraction,Cash and cash equivalents ...
7,SmartCentres,SRU-UN.TO,2023-12-31,FY,current_debt,605.478000,CAD million,2024-SmartCentres-REIT-Annual-Report-.pdf,92,Current portion of debt,605478.00,CAD thousand,Text-based extraction,Current portion of debt ...
8,SmartCentres,SRU-UN.TO,2023-12-31,FY,debt_to_aggregate_assets,43.100000,Percent,2024-SmartCentres-REIT-Annual-Report-.pdf,69,Debt to aggregate assets,43.10,percent,Manual transcription from extracted text,NaN
9,SmartCentres,SRU-UN.TO,2023-12-31,FY,distributions_per_unit,1.850000,CAD/unit,2024-SmartCentres-REIT-Annual-Report-.pdf,10,Distributions declared per unit,1.85,CAD per unit,Manual transcription from extracted text,NaN


In [15]:
# Standardize date format

financial_integrated["period_end"] = pd.to_datetime(
    financial_integrated["period_end"],
    errors="raise"
)

# Define composite primary key

financial_pk = [
    "ticker",
    "period_end",
    "period_type",
    "metric"
]

# Check duplicate primary keys

duplicate_count = financial_integrated.duplicated(
    subset=financial_pk
).sum()

# Check required fields

required_columns = financial_pk + [
    "value",
    "unit",
    "source_file"
]

missing_required = financial_integrated[
    required_columns
].isna().sum()

print("Total records:", len(financial_integrated))
print("Duplicate primary keys:", duplicate_count)

print("\nMissing required values:")
print(missing_required)

print("\nRecords by company and period type:")

display(pd.crosstab(
    financial_integrated["ticker"],
    financial_integrated["period_type"],
    margins=True
))

print("\nReporting period:")
print(financial_integrated["period_end"].min())
print(financial_integrated["period_end"].max())

Total records: 452
Duplicate primary keys: 0

Missing required values:
ticker         0
period_end     0
period_type    0
metric         0
value          0
unit           0
source_file    0
dtype: int64

Records by company and period type:


period_type,FY,Quarterly,All
ticker,,,
CHP-UN.TO,83,48,131
REI-UN.TO,94,57,151
SRU-UN.TO,134,36,170
All,311,141,452



Reporting period:
2023-12-31 00:00:00
2026-09-30 00:00:00


## 3. Financial Metric Harmonization

Identify common financial metrics across SmartCentres,
RioCan, and Choice Properties.

Evaluate metric names, measurement units, reporting
frequencies, and accounting definitions to support
consistent cross-company financial analysis.

Preserve original source information and avoid
unsupported assumptions about metric comparability.

In [18]:
# Inspect financial metrics by company

metric_inventory = (
    financial_integrated
    .groupby("metric")
    .agg(
        company_count=("ticker", "nunique"),
        total_records=("metric", "size"),
        units=("unit", lambda x: ", ".join(sorted(x.unique()))),
        companies=("ticker", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
    .sort_values(
        ["company_count", "metric"],
        ascending=[False, True]
    )
)

print("Total unique financial metrics:",
      financial_integrated["metric"].nunique())

display(metric_inventory)

Total unique financial metrics: 62


,metric,company_count,total_records,units,companies
2,affo,3,9,CAD million,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
3,affo_payout_ratio,3,9,Percent,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
4,affo_per_unit_diluted,3,9,CAD/unit,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
7,book_debt_to_assets_ratio,3,15,Percent,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
8,cash_and_cash_equivalents,3,18,CAD million,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
15,ffo,3,9,CAD million,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
17,ffo_per_unit_diluted,3,9,CAD/unit,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
19,financing_cash_flow,3,9,CAD million,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
24,investing_cash_flow,3,9,CAD million,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"
25,investment_properties,3,18,CAD million,"CHP-UN.TO, REI-UN.TO, SRU-UN.TO"


In [20]:
# Create financial metric coverage matrix

metric_coverage = pd.crosstab(
    financial_integrated["metric"],
    financial_integrated["ticker"]
)

metric_coverage["Company_Count"] = (
    metric_coverage.gt(0).sum(axis=1)
)

metric_coverage = metric_coverage.sort_values(
    ["Company_Count"],
    ascending=False
)

display(
    metric_coverage.style.background_gradient(
        cmap="Greens",
        axis=None
    )
)

ticker,CHP-UN.TO,REI-UN.TO,SRU-UN.TO,Company_Count
metric,,,,
financing_cash_flow,3,3,3,3
total_debt,6,6,3,3
investment_properties,6,6,6,3
same_property_noi_growth,2,2,3,3
property_operating_costs,6,6,6,3
operating_cash_flow,3,3,3,3
ffo_per_unit_diluted,3,3,3,3
net_income,6,6,6,3
ffo,3,3,3,3


In [22]:
# Check measurement unit consistency across datasets

unit_check = (
    financial_integrated
    .groupby("metric")["unit"]
    .nunique()
    .reset_index(name="unit_count")
)

unit_conflicts = unit_check[
    unit_check["unit_count"] > 1
]

print("Total metrics checked:", len(unit_check))
print("Metrics with unit conflicts:", len(unit_conflicts))

display(unit_conflicts)

if not unit_conflicts.empty:
    display(
        financial_integrated.loc[
            financial_integrated["metric"].isin(
                unit_conflicts["metric"]
            ),
            ["ticker", "metric", "unit"]
        ]
        .drop_duplicates()
        .sort_values(["metric", "ticker"])
    )

Total metrics checked: 62
Metrics with unit conflicts: 0


,metric,unit_count


In [24]:
# Identify common financial metrics by reporting frequency

for period in ["FY", "Quarterly"]:

    period_data = financial_integrated[
        financial_integrated["period_type"] == period
    ]

    coverage = (
        period_data
        .groupby("metric")["ticker"]
        .nunique()
    )

    common_metrics = coverage[
        coverage == 3
    ].index.tolist()

    print("=" * 60)
    print("Reporting period:", period)
    print("Common metrics:", len(common_metrics))

    for metric in common_metrics:
        print("-", metric)

Reporting period: FY
Common metrics: 21
- affo
- affo_payout_ratio
- affo_per_unit_diluted
- book_debt_to_assets_ratio
- cash_and_cash_equivalents
- ffo
- ffo_per_unit_diluted
- financing_cash_flow
- investing_cash_flow
- investment_properties
- net_income
- net_rental_income_and_other
- net_rental_income_margin
- operating_cash_flow
- property_operating_costs
- rental_revenue_and_other
- same_property_noi_growth
- total_assets
- total_debt
- total_equity
- total_liabilities
Reporting period: Quarterly
Common metrics: 7
- cash_and_cash_equivalents
- investment_properties
- net_income
- property_operating_costs
- total_assets
- total_equity
- total_liabilities


In [26]:
# Review potentially comparable metric definitions

review_metrics = [
    "rental_revenue",
    "rental_revenue_and_other",
    "net_rental_income",
    "net_rental_income_and_other",
    "total_debt",
    "book_debt_to_assets_ratio"
]

metric_definition_review = (
    financial_integrated.loc[
        financial_integrated["metric"].isin(review_metrics),
        [
            "ticker",
            "period_type",
            "metric",
            "unit",
            "source_label",
            "extraction_method"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["metric", "ticker", "period_type"]
    )
    .reset_index(drop=True)
)

display(metric_definition_review)

,ticker,period_type,metric,unit,source_label,extraction_method
0,CHP-UN.TO,FY,book_debt_to_assets_ratio,Percent,total_debt / total_assets * 100,Calculated
1,CHP-UN.TO,Quarterly,book_debt_to_assets_ratio,Percent,total_debt / total_assets * 100,Calculated
2,REI-UN.TO,FY,book_debt_to_assets_ratio,Percent,total_debt / total_assets * 100,Calculated
3,REI-UN.TO,Quarterly,book_debt_to_assets_ratio,Percent,total_debt / total_assets * 100,Calculated
4,SRU-UN.TO,FY,book_debt_to_assets_ratio,Percent,total_debt / total_assets * 100,Calculated
5,CHP-UN.TO,Quarterly,net_rental_income,CAD million,rental_revenue + property_operating_costs,Calculated
6,REI-UN.TO,Quarterly,net_rental_income,CAD million,rental_revenue + property_operating_costs,Calculated
7,CHP-UN.TO,FY,net_rental_income_and_other,CAD million,rental_revenue_and_other + property_operating_...,Calculated
8,REI-UN.TO,FY,net_rental_income_and_other,CAD million,rental_revenue_and_other + recoverable_operati...,Calculated
9,SRU-UN.TO,FY,net_rental_income_and_other,CAD million,Net rental income and other,Text-based extraction


In [28]:
# Preserve original financial metric names

financial_harmonized = financial_integrated.copy()

metric_mapping = {
    "rental_revenue": "rental_revenue",
    "rental_revenue_and_other": "rental_revenue",
    "net_rental_income": "net_rental_income",
    "net_rental_income_and_other": "net_rental_income"
}

financial_harmonized["metric_group"] = (
    financial_harmonized["metric"]
    .replace(metric_mapping)
)

print("Original metric count:",
      financial_harmonized["metric"].nunique())

print("Harmonized metric group count:",
      financial_harmonized["metric_group"].nunique())

print("Dataset shape:",
      financial_harmonized.shape)

display(
    financial_harmonized[
        ["metric", "metric_group"]
    ]
    .drop_duplicates()
    .query("metric != metric_group")
)

Original metric count: 62
Harmonized metric group count: 60
Dataset shape: (452, 15)


,metric,metric_group
22,net_rental_income_and_other,net_rental_income
33,rental_revenue_and_other,rental_revenue


In [30]:
# Document financial metric comparability limitations

comparability_notes = {
    "rental_revenue":
        "Revenue scope may differ across companies.",

    "net_rental_income":
        "Rental income definitions may differ across companies.",

    "total_debt":
        "Debt components and liability classifications may differ.",

    "book_debt_to_assets_ratio":
        "Comparability depends on the definition of total debt.",

    "ffo":
        "Non-IFRS measure; company definitions may differ.",

    "affo":
        "Non-IFRS measure; company definitions may differ.",

    "ffo_per_unit_diluted":
        "FFO definitions and diluted unit calculations may differ.",

    "affo_per_unit_diluted":
        "AFFO definitions and diluted unit calculations may differ.",

    "affo_payout_ratio":
        "Payout ratio depends on the company-specific AFFO definition.",

    "ffo_payout_ratio":
        "Payout ratio depends on the company-specific FFO definition.",

    "net_rental_income_margin":
        "Revenue and operating cost definitions may differ.",

    "same_property_noi_growth":
        "Same-property portfolio and NOI definitions may differ.",

    "occupancy_rate_committed":
        "Committed occupancy; retain separately.",

    "occupancy_rate_in_place":
        "In-place occupancy; retain separately.",

    "occupancy_rate_reported":
        "Company-reported occupancy; definition requires review.",

    "noi_accounting_basis":
        "Accounting-basis NOI; retain separately.",

    "noi_proportionate":
        "Proportionate-basis NOI; retain separately."
}

financial_harmonized["comparability_note"] = (
    financial_harmonized["metric_group"]
    .map(comparability_notes)
    .fillna(
        "Original definition retained; comparability not independently verified."
    )
)

print("Dataset shape:", financial_harmonized.shape)

display(
    financial_harmonized[
        ["metric", "metric_group", "comparability_note"]
    ]
    .drop_duplicates()
    .head(20)
)

Dataset shape: (452, 16)


,metric,metric_group,comparability_note
0,adjusted_debt_to_adjusted_ebitda,adjusted_debt_to_adjusted_ebitda,Original definition retained; comparability no...
1,affo,affo,Non-IFRS measure; company definitions may differ.
2,affo_payout_ratio,affo_payout_ratio,Payout ratio depends on the company-specific A...
3,affo_per_unit_diluted,affo_per_unit_diluted,AFFO definitions and diluted unit calculations...
4,affo_with_adjustments,affo_with_adjustments,Original definition retained; comparability no...
5,book_debt_to_assets_ratio,book_debt_to_assets_ratio,Comparability depends on the definition of tot...
6,cash_and_cash_equivalents,cash_and_cash_equivalents,Original definition retained; comparability no...
7,current_debt,current_debt,Original definition retained; comparability no...
8,debt_to_aggregate_assets,debt_to_aggregate_assets,Original definition retained; comparability no...
9,distributions_per_unit,distributions_per_unit,Original definition retained; comparability no...


In [32]:
# Validate harmonized financial data

harmonized_pk = [
    "ticker",
    "period_end",
    "period_type",
    "metric_group"
]

group_duplicates = financial_harmonized[
    financial_harmonized.duplicated(
        subset=harmonized_pk,
        keep=False
    )
]

harmonized_unit_conflicts = (
    financial_harmonized
    .groupby("metric_group")["unit"]
    .nunique()
)

harmonized_unit_conflicts = (
    harmonized_unit_conflicts[
        harmonized_unit_conflicts > 1
    ]
)

print("Total records:", len(financial_harmonized))

print("Original primary key duplicates:",
      financial_harmonized.duplicated(financial_pk).sum())

print("Metric group duplicate records:",
      len(group_duplicates))

print("Metric group unit conflicts:",
      len(harmonized_unit_conflicts))

print("\nCommon metric groups by reporting period:")

for period in ["FY", "Quarterly"]:

    subset = financial_harmonized[
        financial_harmonized["period_type"] == period
    ]

    coverage = (
        subset.groupby("metric_group")["ticker"]
        .nunique()
    )

    common = coverage[coverage == 3]

    print("\nPeriod:", period)
    print("Common metric groups:", len(common))

    for metric in common.index:
        print("-", metric)

if not group_duplicates.empty:
    display(group_duplicates)

if not harmonized_unit_conflicts.empty:
    display(harmonized_unit_conflicts)

Total records: 452
Original primary key duplicates: 0
Metric group duplicate records: 0
Metric group unit conflicts: 0

Common metric groups by reporting period:

Period: FY
Common metric groups: 21
- affo
- affo_payout_ratio
- affo_per_unit_diluted
- book_debt_to_assets_ratio
- cash_and_cash_equivalents
- ffo
- ffo_per_unit_diluted
- financing_cash_flow
- investing_cash_flow
- investment_properties
- net_income
- net_rental_income
- net_rental_income_margin
- operating_cash_flow
- property_operating_costs
- rental_revenue
- same_property_noi_growth
- total_assets
- total_debt
- total_equity
- total_liabilities

Period: Quarterly
Common metric groups: 9
- cash_and_cash_equivalents
- investment_properties
- net_income
- net_rental_income
- property_operating_costs
- rental_revenue
- total_assets
- total_equity
- total_liabilities


## 4. Market and Macroeconomic Data Integration

Integrate historical market data with Canadian
interest rates, inflation, and retail sales indicators.

Preserve the original observation frequencies and
validate temporal alignment across datasets.

The integrated data will support investment analysis
and relational database implementation.

In [35]:
# Load cleaned market and macroeconomic datasets

market = pd.read_csv(
    PROCESSED_DIR / "market_data_cleaned.csv"
)

interest_rates = pd.read_csv(
    PROCESSED_DIR / "interest_rates_cleaned.csv"
)

interest_observed = pd.read_csv(
    PROCESSED_DIR / "interest_rates_observed.csv"
)

cpi = pd.read_csv(
    PROCESSED_DIR / "cpi_cleaned.csv"
)

retail_sales = pd.read_csv(
    PROCESSED_DIR / "retail_sales_cleaned.csv"
)

macro_datasets = {
    "Market Data": market,
    "Interest Rates": interest_rates,
    "Observed Interest Rates": interest_observed,
    "CPI": cpi,
    "Retail Sales": retail_sales
}

for name, df in macro_datasets.items():
    print(name, df.shape)

Market Data (3764, 13)
Interest Rates (973, 7)
Observed Interest Rates (973, 4)
CPI (45, 7)
Retail Sales (45, 11)


In [37]:
# Inspect dataset schemas

for name, df in macro_datasets.items():

    print("=" * 60)
    print(name)
    print("Shape:", df.shape)

    display(
        pd.DataFrame({
            "Column": df.columns,
            "Data_Type": df.dtypes.astype(str).values,
            "Missing_Values": df.isna().sum().values
        })
    )

Market Data
Shape: (3764, 13)


,Column,Data_Type,Missing_Values
0,date,object,0
1,open,float64,0
2,high,float64,0
3,low,float64,0
4,close,float64,0
5,adj_close,float64,0
6,volume,int64,0
7,dividends,float64,0
8,stock_splits,float64,0
9,company,object,0


Interest Rates
Shape: (973, 7)


,Column,Data_Type,Missing_Values
0,date,object,0
1,policy_interest_rate,float64,0
2,canada_5y_bond_yield,float64,0
3,canada_10y_bond_yield,float64,0
4,policy_interest_rate_imputed,bool,0
5,canada_5y_bond_yield_imputed,bool,0
6,canada_10y_bond_yield_imputed,bool,0


Observed Interest Rates
Shape: (973, 4)


,Column,Data_Type,Missing_Values
0,date,object,0
1,policy_interest_rate,float64,1
2,canada_5y_bond_yield,float64,39
3,canada_10y_bond_yield,float64,39


CPI
Shape: (45, 7)


,Column,Data_Type,Missing_Values
0,date,object,0
1,geography,object,0
2,cpi_index,float64,0
3,base_year,object,0
4,source_vector,object,0
5,inflation_yoy_pct,float64,0
6,inflation_mom_pct,float64,0


Retail Sales
Shape: (45, 11)


,Column,Data_Type,Missing_Values
0,date,object,0
1,geography,object,0
2,retail_sales_cad_million,float64,0
3,unit,object,0
4,scalar_factor,object,0
5,source_vector,object,0
6,quality_status,object,0
7,currency,object,0
8,seasonally_adjusted,bool,0
9,retail_sales_yoy_pct,float64,0


In [39]:
# Preview market and macroeconomic datasets

for name, df in macro_datasets.items():

    print("=" * 60)
    print(name)

    display(df.head(3))

Market Data


,date,open,high,low,close,adj_close,volume,dividends,stock_splits,company,ticker,currency,capital_gains
0,2023-01-03,14.79,14.97,14.750,14.91,12.140608,537900,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
1,2023-01-04,14.99,15.09,14.910,15.00,12.213893,271300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
2,2023-01-05,14.94,14.94,14.765,14.85,12.091756,472200,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN


Interest Rates


,date,policy_interest_rate,canada_5y_bond_yield,canada_10y_bond_yield,policy_interest_rate_imputed,canada_5y_bond_yield_imputed,canada_10y_bond_yield_imputed
0,2023-01-03,4.25,3.34,3.21,False,False,False
1,2023-01-04,4.25,3.28,3.14,False,False,False
2,2023-01-05,4.25,3.32,3.17,False,False,False


Observed Interest Rates


,date,policy_interest_rate,canada_5y_bond_yield,canada_10y_bond_yield
0,2023-01-03,4.25,3.34,3.21
1,2023-01-04,4.25,3.28,3.14
2,2023-01-05,4.25,3.32,3.17


CPI


,date,geography,cpi_index,base_year,source_vector,inflation_yoy_pct,inflation_mom_pct
0,2023-01-01,Canada,153.9,2002=100,v41690973,5.918789,0.522534
1,2023-02-01,Canada,154.5,2002=100,v41690973,5.245232,0.389864
2,2023-03-01,Canada,155.3,2002=100,v41690973,4.298187,0.517799


Retail Sales


,date,geography,retail_sales_cad_million,unit,scalar_factor,source_vector,quality_status,currency,seasonally_adjusted,retail_sales_yoy_pct,retail_sales_mom_pct
0,2023-01-01,Canada,66337.0,Dollars,millions,v1446870151,A,CAD,True,5.460876,2.492120
1,2023-02-01,Canada,65394.0,Dollars,millions,v1446870151,A,CAD,True,3.428969,-1.421529
2,2023-03-01,Canada,65152.0,Dollars,millions,v1446870151,A,CAD,True,1.784096,-0.370065


In [41]:
# Validate market data primary key

market["date"] = pd.to_datetime(
    market["date"],
    errors="raise"
)

market_pk = ["ticker", "date"]

print("Market records:", len(market))

print("Unique securities:",
      market["ticker"].nunique())

print("Duplicate primary keys:",
      market.duplicated(market_pk).sum())

print("\nRecords by security:")

display(
    market.groupby("ticker")
    .size()
    .reset_index(name="records")
)

print("\nDate range:")

print("Start:", market["date"].min())
print("End:", market["date"].max())

Market records: 3764
Unique securities: 4
Duplicate primary keys: 0

Records by security:


,ticker,records
0,CHP-UN.TO,941
1,REI-UN.TO,941
2,SRU-UN.TO,941
3,XRE.TO,941



Date range:
Start: 2023-01-03 00:00:00
End: 2026-09-30 00:00:00


In [43]:
# Standardize date fields

for df in [
    market,
    interest_rates,
    interest_observed,
    cpi,
    retail_sales
]:
    df["date"] = pd.to_datetime(
        df["date"],
        errors="raise"
    )

# Validate date keys

for name, df in macro_datasets.items():

    key = ["ticker", "date"] if name == "Market Data" else ["date"]

    duplicates = df.duplicated(key).sum()
    missing_keys = df[key].isna().sum().sum()

    print(name)
    print("Duplicate keys:", duplicates)
    print("Missing keys:", missing_keys)
    print("-" * 40)

    assert duplicates == 0
    assert missing_keys == 0

Market Data
Duplicate keys: 0
Missing keys: 0
----------------------------------------
Interest Rates
Duplicate keys: 0
Missing keys: 0
----------------------------------------
Observed Interest Rates
Duplicate keys: 0
Missing keys: 0
----------------------------------------
CPI
Duplicate keys: 0
Missing keys: 0
----------------------------------------
Retail Sales
Duplicate keys: 0
Missing keys: 0
----------------------------------------


In [45]:
# Join daily market data with daily interest rates

market_interest = market.merge(
    interest_rates,
    on="date",
    how="left",
    validate="many_to_one",
    indicator="interest_match"
)

print("Original market shape:", market.shape)
print("Integrated shape:", market_interest.shape)

print("\nInterest rate matching results:")

display(
    market_interest["interest_match"].value_counts()
)

assert len(market_interest) == len(market)

display(market_interest.head())

Original market shape: (3764, 13)
Integrated shape: (3764, 20)

Interest rate matching results:


interest_match
both          3764
left_only        0
right_only       0
Name: count, dtype: int64

,date,open,high,low,close,adj_close,volume,dividends,stock_splits,company,ticker,currency,capital_gains,policy_interest_rate,canada_5y_bond_yield,canada_10y_bond_yield,policy_interest_rate_imputed,canada_5y_bond_yield_imputed,canada_10y_bond_yield_imputed,interest_match
0,2023-01-03,14.79,14.97,14.750,14.91,12.140608,537900,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.34,3.21,False,False,False,both
1,2023-01-04,14.99,15.09,14.910,15.00,12.213893,271300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.28,3.14,False,False,False,both
2,2023-01-05,14.94,14.94,14.765,14.85,12.091756,472200,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.32,3.17,False,False,False,both
3,2023-01-06,14.90,15.04,14.830,14.95,12.173180,406300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.24,3.09,False,False,False,both
4,2023-01-09,14.87,15.10,14.790,14.94,12.165037,290300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.22,3.08,False,False,False,both


In [47]:
# Rename source-specific metadata columns

cpi_monthly = cpi.rename(columns={
    "source_vector": "cpi_source_vector"
})

retail_monthly = retail_sales.rename(columns={
    "source_vector": "retail_source_vector",
    "unit": "retail_unit",
    "scalar_factor": "retail_scalar_factor",
    "quality_status": "retail_quality_status",
    "currency": "retail_currency"
})

# Integrate monthly macroeconomic datasets

monthly_macro = cpi_monthly.merge(
    retail_monthly,
    on=["date", "geography"],
    how="outer",
    validate="one_to_one",
    indicator="macro_match"
)

print("CPI shape:", cpi.shape)
print("Retail sales shape:", retail_sales.shape)
print("Monthly macroeconomic shape:", monthly_macro.shape)

print("\nMatching results:")

display(
    monthly_macro["macro_match"].value_counts()
)

display(monthly_macro.head())

CPI shape: (45, 7)
Retail sales shape: (45, 11)
Monthly macroeconomic shape: (45, 17)

Matching results:


macro_match
both          45
left_only      0
right_only     0
Name: count, dtype: int64

,date,geography,cpi_index,base_year,cpi_source_vector,inflation_yoy_pct,inflation_mom_pct,retail_sales_cad_million,retail_unit,retail_scalar_factor,retail_source_vector,retail_quality_status,retail_currency,seasonally_adjusted,retail_sales_yoy_pct,retail_sales_mom_pct,macro_match
0,2023-01-01,Canada,153.9,2002=100,v41690973,5.918789,0.522534,66337.0,Dollars,millions,v1446870151,A,CAD,True,5.460876,2.492120,both
1,2023-02-01,Canada,154.5,2002=100,v41690973,5.245232,0.389864,65394.0,Dollars,millions,v1446870151,A,CAD,True,3.428969,-1.421529,both
2,2023-03-01,Canada,155.3,2002=100,v41690973,4.298187,0.517799,65152.0,Dollars,millions,v1446870151,A,CAD,True,1.784096,-0.370065,both
3,2023-04-01,Canada,156.4,2002=100,v41690973,4.405874,0.708307,65544.0,Dollars,millions,v1446870151,A,CAD,True,2.278260,0.601670,both
4,2023-05-01,Canada,157.0,2002=100,v41690973,3.357472,0.383632,65912.0,Dollars,millions,v1446870151,A,CAD,True,-0.121227,0.561455,both


In [49]:
# Create monthly alignment keys

market_interest["month"] = (
    market_interest["date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly_macro["month"] = (
    monthly_macro["date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Select macroeconomic indicators for analysis

macro_indicators = monthly_macro[
    [
        "month",
        "cpi_index",
        "inflation_yoy_pct",
        "inflation_mom_pct",
        "retail_sales_cad_million",
        "retail_sales_yoy_pct",
        "retail_sales_mom_pct"
    ]
].copy()

# Join market data with monthly macroeconomic indicators

market_macro_integrated = market_interest.merge(
    macro_indicators,
    on="month",
    how="left",
    validate="many_to_one",
    indicator="monthly_match"
)

print("Integrated market and macro shape:",
      market_macro_integrated.shape)

print("\nMonthly macro matching results:")

display(
    market_macro_integrated["monthly_match"].value_counts()
)

display(market_macro_integrated.head())

Integrated market and macro shape: (3764, 28)

Monthly macro matching results:


monthly_match
both          3764
left_only        0
right_only       0
Name: count, dtype: int64

,date,open,high,low,close,adj_close,volume,dividends,stock_splits,company,ticker,currency,capital_gains,policy_interest_rate,canada_5y_bond_yield,canada_10y_bond_yield,policy_interest_rate_imputed,canada_5y_bond_yield_imputed,canada_10y_bond_yield_imputed,interest_match,month,cpi_index,inflation_yoy_pct,inflation_mom_pct,retail_sales_cad_million,retail_sales_yoy_pct,retail_sales_mom_pct,monthly_match
0,2023-01-03,14.79,14.97,14.750,14.91,12.140608,537900,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.34,3.21,False,False,False,both,2023-01-01,153.9,5.918789,0.522534,66337.0,5.460876,2.49212,both
1,2023-01-04,14.99,15.09,14.910,15.00,12.213893,271300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.28,3.14,False,False,False,both,2023-01-01,153.9,5.918789,0.522534,66337.0,5.460876,2.49212,both
2,2023-01-05,14.94,14.94,14.765,14.85,12.091756,472200,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.32,3.17,False,False,False,both,2023-01-01,153.9,5.918789,0.522534,66337.0,5.460876,2.49212,both
3,2023-01-06,14.90,15.04,14.830,14.95,12.173180,406300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.24,3.09,False,False,False,both,2023-01-01,153.9,5.918789,0.522534,66337.0,5.460876,2.49212,both
4,2023-01-09,14.87,15.10,14.790,14.94,12.165037,290300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN,4.25,3.22,3.08,False,False,False,both,2023-01-01,153.9,5.918789,0.522534,66337.0,5.460876,2.49212,both


In [51]:
# Validate the integrated market and macroeconomic dataset

macro_columns = [
    "policy_interest_rate",
    "canada_5y_bond_yield",
    "canada_10y_bond_yield",
    "cpi_index",
    "inflation_yoy_pct",
    "inflation_mom_pct",
    "retail_sales_cad_million",
    "retail_sales_yoy_pct",
    "retail_sales_mom_pct"
]

print("Original market records:", len(market))

print("Integrated records:",
      len(market_macro_integrated))

print("Duplicate primary keys:",
      market_macro_integrated.duplicated(
          ["ticker", "date"]
      ).sum())

print("\nMissing macroeconomic values:")

display(
    market_macro_integrated[macro_columns]
    .isna()
    .sum()
)

print("\nUnique months:",
      market_macro_integrated["month"].nunique())

print("\nUnique securities:",
      market_macro_integrated["ticker"].nunique())

assert len(market_macro_integrated) == len(market)

assert not market_macro_integrated.duplicated(
    ["ticker", "date"]
).any()

assert market_macro_integrated[
    "interest_match"
].eq("both").all()

assert market_macro_integrated[
    "monthly_match"
].eq("both").all()

assert not market_macro_integrated[
    macro_columns
].isna().any().any()

print("\nMarket and macroeconomic integration: PASSED")

Original market records: 3764
Integrated records: 3764
Duplicate primary keys: 0

Missing macroeconomic values:


policy_interest_rate        0
canada_5y_bond_yield        0
canada_10y_bond_yield       0
cpi_index                   0
inflation_yoy_pct           0
inflation_mom_pct           0
retail_sales_cad_million    0
retail_sales_yoy_pct        0
retail_sales_mom_pct        0
dtype: int64


Unique months: 45

Unique securities: 4

Market and macroeconomic integration: PASSED


## 5. Database Table Preparation

Prepare normalized, database-ready datasets for MySQL.

Separate security information, financial metric definitions,
financial observations, market prices, daily interest rates,
and monthly macroeconomic indicators.

Preserve primary keys, source references, measurement units,
and data quality information.

In [54]:
# Create securities master table

securities = (
    market[["ticker", "company", "currency"]]
    .drop_duplicates()
    .rename(columns={"company": "security_name"})
    .sort_values("ticker")
    .reset_index(drop=True)
)

securities["security_type"] = np.where(
    securities["ticker"] == "XRE.TO",
    "ETF",
    "REIT"
)

securities["analysis_role"] = securities["ticker"].map({
    "SRU-UN.TO": "Target",
    "REI-UN.TO": "Peer",
    "CHP-UN.TO": "Peer",
    "XRE.TO": "Benchmark"
})

assert securities["ticker"].is_unique
assert securities["analysis_role"].notna().all()

print("Securities table shape:", securities.shape)

display(securities)

Securities table shape: (4, 5)


,ticker,security_name,currency,security_type,analysis_role
0,CHP-UN.TO,Choice Properties,CAD,REIT,Peer
1,REI-UN.TO,RioCan,CAD,REIT,Peer
2,SRU-UN.TO,SmartCentres,CAD,REIT,Target
3,XRE.TO,REIT Benchmark,CAD,ETF,Benchmark


In [56]:
# Create financial metric definitions table

metric_definitions = (
    financial_harmonized[
        [
            "metric",
            "metric_group",
            "unit",
            "comparability_note"
        ]
    ]
    .drop_duplicates()
    .sort_values("metric")
    .reset_index(drop=True)
)

assert metric_definitions["metric"].is_unique

print("Metric definitions table shape:",
      metric_definitions.shape)

display(metric_definitions.head(10))

Metric definitions table shape: (62, 4)


,metric,metric_group,unit,comparability_note
0,adjusted_debt_to_adjusted_ebitda,adjusted_debt_to_adjusted_ebitda,Times,Original definition retained; comparability no...
1,adjusted_debt_to_total_assets,adjusted_debt_to_total_assets,Percent,Original definition retained; comparability no...
2,affo,affo,CAD million,Non-IFRS measure; company definitions may differ.
3,affo_payout_ratio,affo_payout_ratio,Percent,Payout ratio depends on the company-specific A...
4,affo_per_unit_diluted,affo_per_unit_diluted,CAD/unit,AFFO definitions and diluted unit calculations...
5,affo_with_adjustments,affo_with_adjustments,CAD million,Original definition retained; comparability no...
6,bank_loans,bank_loans,CAD million,Original definition retained; comparability no...
7,book_debt_to_assets_ratio,book_debt_to_assets_ratio,Percent,Comparability depends on the definition of tot...
8,cash_and_cash_equivalents,cash_and_cash_equivalents,CAD million,Original definition retained; comparability no...
9,credit_facility,credit_facility,CAD million,Original definition retained; comparability no...


In [58]:
# Create financial observations table

financial_metrics = (
    financial_harmonized
    .drop(columns=[
        "company",
        "metric_group",
        "unit",
        "comparability_note"
    ])
    .sort_values([
        "ticker",
        "period_end",
        "period_type",
        "metric"
    ])
    .reset_index(drop=True)
)

print("Financial metrics table shape:",
      financial_metrics.shape)

display(financial_metrics.head())

Financial metrics table shape: (452, 12)


,ticker,period_end,period_type,metric,value,source_file,source_page,source_label,original_value,original_unit,extraction_method,original_line
0,CHP-UN.TO,2023-12-31,FY,affo,598.432000,Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf,72,Adjusted Funds from Operations,598432.000,CAD thousand,Manual transcription from extracted text,NaN
1,CHP-UN.TO,2023-12-31,FY,affo_payout_ratio,90.500000,Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf,72,AFFO payout ratio - diluted,90.500,Percent,Manual transcription from extracted text,NaN
2,CHP-UN.TO,2023-12-31,FY,affo_per_unit_diluted,0.827000,Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf,72,AFFO per unit diluted,0.827,CAD/unit,Manual transcription from extracted text,NaN
3,CHP-UN.TO,2023-12-31,FY,book_debt_to_assets_ratio,38.685242,Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf,109,total_debt / total_assets * 100,NaN,NaN,Calculated,total_debt / total_assets * 100
4,CHP-UN.TO,2023-12-31,FY,cash_and_cash_equivalents,252.424000,Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf,109,Cash and cash equivalents,252424.000,CAD thousand,Text-based extraction,"Cash and cash equivalents 29(c) 63,388 252,424"


In [60]:
# Create daily market prices table

market_prices = (
    market
    .drop(columns=["company", "currency"])
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

print("Market prices table shape:",
      market_prices.shape)

display(market_prices.head())

Market prices table shape: (3764, 11)


,date,open,high,low,close,adj_close,volume,dividends,stock_splits,ticker,capital_gains
0,2023-01-03,14.79,14.97,14.750,14.91,12.140608,537900,0.0,0.0,CHP-UN.TO,NaN
1,2023-01-04,14.99,15.09,14.910,15.00,12.213893,271300,0.0,0.0,CHP-UN.TO,NaN
2,2023-01-05,14.94,14.94,14.765,14.85,12.091756,472200,0.0,0.0,CHP-UN.TO,NaN
3,2023-01-06,14.90,15.04,14.830,14.95,12.173180,406300,0.0,0.0,CHP-UN.TO,NaN
4,2023-01-09,14.87,15.10,14.790,14.94,12.165037,290300,0.0,0.0,CHP-UN.TO,NaN


In [62]:
# Preserve observed and cleaned interest rate values

observed_rates = interest_observed.rename(columns={
    "policy_interest_rate":
        "policy_interest_rate_observed",

    "canada_5y_bond_yield":
        "canada_5y_bond_yield_observed",

    "canada_10y_bond_yield":
        "canada_10y_bond_yield_observed"
})

daily_interest_rates = (
    interest_rates.merge(
        observed_rates,
        on="date",
        how="left",
        validate="one_to_one"
    )
    .sort_values("date")
    .reset_index(drop=True)
)

print("Daily interest rates table shape:",
      daily_interest_rates.shape)

display(daily_interest_rates.head())

Daily interest rates table shape: (973, 10)


,date,policy_interest_rate,canada_5y_bond_yield,canada_10y_bond_yield,policy_interest_rate_imputed,canada_5y_bond_yield_imputed,canada_10y_bond_yield_imputed,policy_interest_rate_observed,canada_5y_bond_yield_observed,canada_10y_bond_yield_observed
0,2023-01-03,4.25,3.34,3.21,False,False,False,4.25,3.34,3.21
1,2023-01-04,4.25,3.28,3.14,False,False,False,4.25,3.28,3.14
2,2023-01-05,4.25,3.32,3.17,False,False,False,4.25,3.32,3.17
3,2023-01-06,4.25,3.24,3.09,False,False,False,4.25,3.24,3.09
4,2023-01-09,4.25,3.22,3.08,False,False,False,4.25,3.22,3.08


In [64]:
# Create monthly macroeconomic table

monthly_macro_db = (
    monthly_macro
    .drop(columns=["macro_match", "date"])
    .sort_values("month")
    .reset_index(drop=True)
)

print("Monthly macro table shape:",
      monthly_macro_db.shape)

display(monthly_macro_db.head())

Monthly macro table shape: (45, 16)


,geography,cpi_index,base_year,cpi_source_vector,inflation_yoy_pct,inflation_mom_pct,retail_sales_cad_million,retail_unit,retail_scalar_factor,retail_source_vector,retail_quality_status,retail_currency,seasonally_adjusted,retail_sales_yoy_pct,retail_sales_mom_pct,month
0,Canada,153.9,2002=100,v41690973,5.918789,0.522534,66337.0,Dollars,millions,v1446870151,A,CAD,True,5.460876,2.492120,2023-01-01
1,Canada,154.5,2002=100,v41690973,5.245232,0.389864,65394.0,Dollars,millions,v1446870151,A,CAD,True,3.428969,-1.421529,2023-02-01
2,Canada,155.3,2002=100,v41690973,4.298187,0.517799,65152.0,Dollars,millions,v1446870151,A,CAD,True,1.784096,-0.370065,2023-03-01
3,Canada,156.4,2002=100,v41690973,4.405874,0.708307,65544.0,Dollars,millions,v1446870151,A,CAD,True,2.278260,0.601670,2023-04-01
4,Canada,157.0,2002=100,v41690973,3.357472,0.383632,65912.0,Dollars,millions,v1446870151,A,CAD,True,-0.121227,0.561455,2023-05-01


## 6. Final Database Validation

Validate primary key uniqueness, referential integrity,
required fields, data coverage, and record counts before
exporting the database-ready datasets.

In [67]:
# Define database tables and primary keys

database_tables = {
    "securities": securities,
    "metric_definitions": metric_definitions,
    "financial_metrics": financial_metrics,
    "market_prices": market_prices,
    "daily_interest_rates": daily_interest_rates,
    "monthly_macro": monthly_macro_db
}

primary_keys = {
    "securities": ["ticker"],

    "metric_definitions": ["metric"],

    "financial_metrics": [
        "ticker",
        "period_end",
        "period_type",
        "metric"
    ],

    "market_prices": ["ticker", "date"],

    "daily_interest_rates": ["date"],

    "monthly_macro": ["month"]
}

validation_results = []

for name, df in database_tables.items():

    pk = primary_keys[name]

    duplicates = df.duplicated(pk).sum()

    missing_keys = df[pk].isna().sum().sum()

    validation_results.append({
        "Table": name,
        "Records": len(df),
        "Columns": len(df.columns),
        "Duplicate_PK": duplicates,
        "Missing_PK": missing_keys,
        "Status": (
            "PASS"
            if duplicates == 0 and missing_keys == 0
            else "FAIL"
        )
    })

validation_summary = pd.DataFrame(validation_results)

display(validation_summary)

assert validation_summary["Status"].eq("PASS").all()

print("Primary key validation: PASSED")

,Table,Records,Columns,Duplicate_PK,Missing_PK,Status
0,securities,4,5,0,0,PASS
1,metric_definitions,62,4,0,0,PASS
2,financial_metrics,452,12,0,0,PASS
3,market_prices,3764,11,0,0,PASS
4,daily_interest_rates,973,10,0,0,PASS
5,monthly_macro,45,16,0,0,PASS


Primary key validation: PASSED


In [69]:
# Validate foreign key relationships

fk_checks = {
    "Financial Metrics -> Securities":
        financial_metrics["ticker"].isin(
            securities["ticker"]
        ).all(),

    "Market Prices -> Securities":
        market_prices["ticker"].isin(
            securities["ticker"]
        ).all(),

    "Financial Metrics -> Metric Definitions":
        financial_metrics["metric"].isin(
            metric_definitions["metric"]
        ).all()
}

fk_validation = pd.DataFrame([
    {
        "Relationship": name,
        "Status": "PASS" if result else "FAIL"
    }
    for name, result in fk_checks.items()
])

display(fk_validation)

assert all(fk_checks.values())

print("Foreign key validation: PASSED")

,Relationship,Status
0,Financial Metrics -> Securities,PASS
1,Market Prices -> Securities,PASS
2,Financial Metrics -> Metric Definitions,PASS


Foreign key validation: PASSED


In [71]:
# Validate temporal join coverage

temporal_checks = {
    "Market dates covered by interest rates":
        market_prices["date"].isin(
            daily_interest_rates["date"]
        ).all(),

    "Market months covered by monthly macro":
        market_prices["date"]
        .dt.to_period("M")
        .dt.to_timestamp()
        .isin(monthly_macro_db["month"])
        .all()
}

temporal_validation = pd.DataFrame([
    {
        "Validation": name,
        "Status": "PASS" if result else "FAIL"
    }
    for name, result in temporal_checks.items()
])

display(temporal_validation)

assert all(temporal_checks.values())

print("Temporal alignment validation: PASSED")

,Validation,Status
0,Market dates covered by interest rates,PASS
1,Market months covered by monthly macro,PASS


Temporal alignment validation: PASSED


In [73]:
# Validate record preservation

expected_counts = {
    "securities": 4,
    "metric_definitions": 62,
    "financial_metrics": 452,
    "market_prices": 3764,
    "daily_interest_rates": 973,
    "monthly_macro": 45
}

record_validation = pd.DataFrame([
    {
        "Table": name,
        "Expected": expected,
        "Actual": len(database_tables[name]),
        "Status": (
            "PASS"
            if len(database_tables[name]) == expected
            else "FAIL"
        )
    }
    for name, expected in expected_counts.items()
])

display(record_validation)

assert record_validation["Status"].eq("PASS").all()

# Confirm required financial values are preserved

assert financial_metrics[
    ["value", "source_file"]
].notna().all().all()

print("Record preservation validation: PASSED")

,Table,Expected,Actual,Status
0,securities,4,4,PASS
1,metric_definitions,62,62,PASS
2,financial_metrics,452,452,PASS
3,market_prices,3764,3764,PASS
4,daily_interest_rates,973,973,PASS
5,monthly_macro,45,45,PASS


Record preservation validation: PASSED


## 7. Export Database-Ready Datasets

Export the validated relational datasets for MySQL
database implementation.

Retain separate analytical datasets for retrospective
market and macroeconomic analysis.

Document the validation results and preserve all
original cleaned datasets.

In [76]:
# Export all database-ready tables

DATABASE_DIR = INTEGRATED_DIR / "database_ready"

DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

export_results = []

for name, df in database_tables.items():

    output_path = DATABASE_DIR / f"{name}.csv"

    export_df = df.copy()

    # Convert Boolean fields for MySQL import

    for col in export_df.select_dtypes(
        include=["bool"]
    ).columns:
        export_df[col] = export_df[col].astype(int)

    export_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
        date_format="%Y-%m-%d"
    )

    reloaded = pd.read_csv(output_path)

    export_passed = (
        reloaded.shape == export_df.shape
        and list(reloaded.columns) == list(export_df.columns)
        and not reloaded.duplicated(primary_keys[name]).any()
        and reloaded[primary_keys[name]].notna().all().all()
    )

    export_results.append({
        "File": output_path.name,
        "Rows": len(reloaded),
        "Columns": len(reloaded.columns),
        "Export_Status": (
            "PASS" if export_passed else "FAIL"
        )
    })

export_summary = pd.DataFrame(export_results)

display(export_summary)

assert export_summary["Export_Status"].eq("PASS").all()

print("All database-ready CSV exports: PASSED")

print("Export directory:", DATABASE_DIR)

,File,Rows,Columns,Export_Status
0,securities.csv,4,5,PASS
1,metric_definitions.csv,62,4,PASS
2,financial_metrics.csv,452,12,PASS
3,market_prices.csv,3764,11,PASS
4,daily_interest_rates.csv,973,10,PASS
5,monthly_macro.csv,45,16,PASS


All database-ready CSV exports: PASSED
Export directory: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/integrated/database_ready


In [78]:
# Export analytical datasets

ANALYSIS_DIR = INTEGRATED_DIR / "analysis"

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

financial_harmonized.to_csv(
    ANALYSIS_DIR / "financial_harmonized.csv",
    index=False
)

market_macro_integrated.to_csv(
    ANALYSIS_DIR / "market_macro_retrospective.csv",
    index=False
)

metric_coverage.to_csv(
    ANALYSIS_DIR / "metric_coverage.csv"
)

# Export validation reports

VALIDATION_DIR = INTEGRATED_DIR / "validation"

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

validation_summary.to_csv(
    VALIDATION_DIR / "primary_key_validation.csv",
    index=False
)

fk_validation.to_csv(
    VALIDATION_DIR / "foreign_key_validation.csv",
    index=False
)

temporal_validation.to_csv(
    VALIDATION_DIR / "temporal_validation.csv",
    index=False
)

record_validation.to_csv(
    VALIDATION_DIR / "record_validation.csv",
    index=False
)

export_summary.to_csv(
    VALIDATION_DIR / "export_validation.csv",
    index=False
)

print("Analytical datasets exported successfully.")
print("Validation reports exported successfully.")

Analytical datasets exported successfully.
Validation reports exported successfully.


In [80]:
# Final data integration summary

print("=" * 65)
print("BSMM 8730 - GROUP 2")
print("DATA INTEGRATION FINAL SUMMARY")
print("=" * 65)

print("Total database tables:", len(database_tables))

print("Total database records:",
      sum(len(df) for df in database_tables.values()))

print("Financial observations:", len(financial_metrics))

print("Market observations:", len(market_prices))

print("Original financial metrics:", len(metric_definitions))

print("Harmonized metric groups:",
      financial_harmonized["metric_group"].nunique())

print("Primary key validation: PASSED")
print("Foreign key validation: PASSED")
print("Temporal validation: PASSED")
print("Record preservation: PASSED")
print("CSV export validation: PASSED")

print("=" * 65)

print("DATA INTEGRATION COMPLETED SUCCESSFULLY")

print("=" * 65)

BSMM 8730 - GROUP 2
DATA INTEGRATION FINAL SUMMARY
Total database tables: 6
Total database records: 5300
Financial observations: 452
Market observations: 3764
Original financial metrics: 62
Harmonized metric groups: 60
Primary key validation: PASSED
Foreign key validation: PASSED
Temporal validation: PASSED
Record preservation: PASSED
CSV export validation: PASSED
DATA INTEGRATION COMPLETED SUCCESSFULLY


## 8. Conclusion and Limitations

The data integration phase prepared six relational datasets
for MySQL database implementation.

The integrated datasets preserve the original financial
observations, market records, macroeconomic indicators,
source references, and data quality information.

Primary key uniqueness, referential integrity, temporal
alignment, and CSV export structure were validated.

### Limitations

- Financial metric groups represent analytical classifications
  rather than guaranteed accounting equivalence.
- Company-specific FFO, AFFO, NOI, occupancy, and debt
  definitions require careful interpretation.
- Missing financial metric coverage is not treated as zero.
- Original missing capital gains and observed interest rate
  values are preserved.
- The retrospective market-macro dataset uses reference-month
  alignment, not actual publication-date alignment.
- The retrospective dataset must not be used directly for
  historical trading simulations or predictive modeling.
- Database constraints and normalization will be verified
  during the MySQL implementation phase.

### Next Phase

Database Design, Entity-Relationship Diagram (ERD),
MySQL Implementation, and SQL-Based Investment Analysis.